# Experiment 3: implementation feasibility

Strategy: the configured DCC-GARCH model with the 60-month calibration rescale (`config/risk_config.json`), primary cost 2 bps, 1976-2014, data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`). Earlier reports under `reports/*_20261026/` used the rolling-covariance model and are left untouched. Logic lives in `scripts/run_implementation_feasibility.py`; this notebook runs it and reads the output in `reports/dcc_garch_studies/feasibility/`.

## Goal

Two checks that do not change the strategy: (1) is reported daily volume large enough to be a usable activity screen for the trades the strategy makes; (2) how much does Sharpe move if cash collateral earns the supplied monthly risk-free rate (2000-2014, fully collateralised, no financing spread). Needs the trade events from experiment 2, so run that notebook first.

## Run

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
REFERENCE = ROOT / "reports/cost_revision_20261026"  # hashed Step 2 inputs (data only)
STUDIES = ROOT / "reports/dcc_garch_studies"
pd.options.display.float_format = "{:.4f}".format
from run_implementation_feasibility import main

OUT = STUDIES / "feasibility"
if not OUT.exists():
    main(REFERENCE, STUDIES / "roll", OUT)

## Volume screen coverage of executed trades

In [2]:
pd.read_csv(OUT / "trade_activity_coverage.csv")

,strategy,asset_class,events,event_coverage,notional_coverage,incomplete,zero_mean_volume,stale
0,trend,ALL,14701,1.0000,1.0000,0,0,0
1,trend,COMM,9365,1.0000,1.0000,0,0,0
2,trend,EQ,1824,1.0000,1.0000,0,0,0
3,trend,FI,1702,1.0000,1.0000,0,0,0
4,trend,FX,1810,1.0000,1.0000,0,0,0
5,static,ALL,12506,1.0000,1.0000,0,0,0
6,static,COMM,7677,1.0000,1.0000,0,0,0
7,static,EQ,1585,1.0000,1.0000,0,0,0
8,static,FI,1455,1.0000,1.0000,0,0,0
9,static,FX,1789,1.0000,1.0000,0,0,0


## Collateral income sensitivity (2000-2014)

In [3]:
funding = pd.read_csv(OUT / "funding_comparison.csv")
funding[["trade_bps", "strategy", "collateral_rf", "cagr", "max_drawdown", "collateral_income_ann", "excess_sharpe_vs_workbook_rf"]]

,trade_bps,strategy,collateral_rf,cagr,max_drawdown,collateral_income_ann,excess_sharpe_vs_workbook_rf
0,1,trend,False,0.0853,-0.0851,0.0000,0.7546
1,1,trend,True,0.1051,-0.0850,0.0183,0.9580
2,1,static,False,0.0732,-0.2848,0.0000,0.5502
3,1,static,True,0.0928,-0.2755,0.0183,0.7267
4,2,trend,False,0.0843,-0.0855,0.0000,0.7443
5,2,trend,True,0.1040,-0.0854,0.0183,0.9478
6,2,static,False,0.0730,-0.2850,0.0000,0.5481
7,2,static,True,0.0926,-0.2758,0.0183,0.7246


## What the data cannot support

In [4]:
gate = json.loads((OUT / "data_gate.json").read_text())
for topic, missing in gate["unsupported"].items():
    print(topic, "->", ", ".join(missing))

capacity_and_participation -> AUM, dated delivery-contract IDs, multipliers and quote units, FX conversion, verified contract volume
market_impact -> spreads, execution horizon, fills/benchmarks, futures-specific coefficient calibration
margin_and_borrowing -> contract margin schedules, cash balances, cash buffers, actual collateral yield and financing terms


## Interpretation

- Every executed trade passes the positive-recent-volume screen, with no stale or incomplete windows, so reported volume does not by itself flag an unusable market. The volumes are reported-series units, not verified contract volume, so this is descriptive only.
- Paying RF on collateral raises 2000-2014 TREND CAGR from 8.4% to 10.4% and the excess Sharpe against RF from 0.74 to 0.95 at 2 bps. The income is the same ~1.8%/yr for TREND and STATIC, so it does not change their ranking.
- Capacity, market impact and margin cannot be assessed without AUM, contract data, spreads and margin schedules. No strategy change is adopted.